In [2]:
import torch
from transformers import GPT2LMHeadModel, DataCollatorForLanguageModeling, TrainingArguments, Trainer, set_seed
import pandas as pd
import numpy as np
from datasets import Dataset, load_from_disk, concatenate_datasets
from transformers import GPT2TokenizerFast
from pathlib import Path
import json

In [3]:
seed = 45
set_seed(seed)
np.random.seed(seed)

In [4]:
out_dir = Path('../data/processed/artifacts/CPT_user_behavior_V1/')

tokenizer = GPT2TokenizerFast.from_pretrained(out_dir / 'tokenizer')

with open(out_dir / 'token_spec.json', 'r', encoding='utf-8') as f:
    token_spec = json.load(f)
    
ds_user_behavior = load_from_disk(out_dir / 'data' / 'behavior')
ds_meta_title = load_from_disk(out_dir / 'data' / 'meta_title')
ds_meta_genre = load_from_disk(out_dir / 'data' / 'meta_genre')
ds_meta_year  = load_from_disk(out_dir / 'data' / 'meta_year')

def sample_len_stats(ds, n=2000):
    m = min(n, len(ds))
    lens = [len(ds[i]['input_ids']) for i in range(m)]
    return {
        'min': int(np.min(lens)),
        'median': int(np.median(lens)),
        'p95': int(np.percentile(lens, 95)),
        'max': int(np.max(lens)),
    }

print('behavior:', sample_len_stats(ds_user_behavior))
print('meta_title:', sample_len_stats(ds_meta_title))
print('meta_genre:', sample_len_stats(ds_meta_genre))
print('meta_year:', sample_len_stats(ds_meta_year))

ratio_behavior = 0.5

ds_meta_all = concatenate_datasets([ds_meta_title, ds_meta_genre, ds_meta_year]).shuffle(seed=seed)
ds_user_behavior = ds_user_behavior.shuffle(seed=seed)

max_total = int(min(len(ds_user_behavior) / ratio_behavior, len(ds_meta_all) / (1 - ratio_behavior)))

n_user_behavior = int(max_total * ratio_behavior)
n_meta_all = int(max_total * (1 - ratio_behavior))

ds_user_behavior_selected = ds_user_behavior.select(range(n_user_behavior))
ds_meta_all_selected = ds_meta_all.select(range(n_meta_all))

ds_cpt = concatenate_datasets([ds_user_behavior_selected, ds_meta_all_selected]).shuffle(seed=seed)

print('ds_cpt size:', len(ds_cpt), 'behavior:', n_user_behavior, 'meta:', n_meta_all)

split = ds_cpt.train_test_split(test_size=0.02, seed=seed)
ds_train = split['train']
ds_val = split['test']

print('train:', len(ds_train), 'val:', len(ds_val))

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


behavior: {'min': 152, 'median': 728, 'p95': 1008, 'max': 1008}
meta_title: {'min': 15, 'median': 17, 'p95': 22, 'max': 33}
meta_genre: {'min': 17, 'median': 20, 'p95': 25, 'max': 33}
meta_year: {'min': 16, 'median': 16, 'p95': 16, 'max': 16}
ds_cpt size: 12080 behavior: 6040 meta: 6040
train: 11838 val: 242


In [5]:
model = GPT2LMHeadModel.from_pretrained('gpt2-large')
model.resize_token_embeddings(len(tokenizer))

model.config.eos_token_id = tokenizer.eos_token_id
model.config.bos_token_id = tokenizer.bos_token_id
model.config.pad_token_id = tokenizer.pad_token_id 

collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

out_model_dir = out_dir / 'cpt_gpt2-large_v1'

args = TrainingArguments(
    output_dir=str(out_model_dir),
    overwrite_output_dir=False,
    num_train_epochs=1,
    learning_rate=9e-5,
    warmup_ratio=0.03,
    weight_decay=0.01,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=32, 
    eval_strategy='steps',
    save_strategy='steps',
    eval_steps=200,
    save_steps=200,
    logging_steps=50,
    bf16=True,
    report_to='none',
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    save_total_limit=1,
    
    
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=ds_train,
    eval_dataset=ds_val,
    data_collator=collator,
    tokenizer=tokenizer,
)

trainer.train()
trainer.save_model(str(out_model_dir))
tokenizer.save_pretrained(str(out_model_dir))

config.json:   0%|          | 0.00/666 [00:00<?, ?B/s]

C:\Users\User\.conda\conda.new\envs\plum2\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\User\.cache\huggingface\hub\models--gpt2-large. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors:   0%|          | 0.00/3.25G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Step,Training Loss,Validation Loss


KeyboardInterrupt: 